In [24]:
import json
import pandas as pd
import re

from bs4 import BeautifulSoup

with open("../data/raw_ads_2511.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

## Datakvalitetskontroll inför städning

In [25]:
print("Antal annonser:", len(df))
print("Unika annons-ID:", df["id"].nunique())

Antal annonser: 11204
Unika annons-ID: 6649


In [26]:
missing_description = df[
    df["description"].isna() |
    df["description"].fillna("").str.strip().eq("")
]

print("Antal annonser utan description:", len(missing_description))

missing_description[
    ["id", "title", "employer", "occupation", "description"]
].head(10)

Antal annonser utan description: 14


,id,title,employer,occupation,description
489,29506534,Teamcenter Solution Architect operations,Hitachi Energy Sweden AB,IT-arkitekt/Lösningsarkitekt,
2247,29737028,Core Network Analyst,Semicon Service Nordic AB,Systemanalytiker/Systemutredare,
2638,29737028,Core Network Analyst,Semicon Service Nordic AB,Systemanalytiker/Systemutredare,
3645,29817808,Senior Power BI Specialist - Älmhult,Danda AB,"Affärskonsult, IT",
3662,29817808,Senior Power BI Specialist - Älmhult,Danda AB,"Affärskonsult, IT",
7113,30456047,UI/UX designer,GreenBay GreenTech Nordic AB,IT-utredare,
7272,30456047,UI/UX designer,GreenBay GreenTech Nordic AB,IT-utredare,
8116,30600803,Dynamics 365 Architect / D365 Solution Archite...,Veritaz AB,Enterprise Architect,
8863,30983013,Linux Engineer,Ericsson AB,Systemanalytiker/Systemutredare,
8903,30961171,Senior Data Engineer,Semicon Service Nordic AB,Data scientist,


In [27]:
duplicate_ids = df[
    df["id"].duplicated(keep=False)
].sort_values("id")

print("Antal rader som tillhör ett duplicerat ID:", len(duplicate_ids))
print("Antal duplicerade ID:", duplicate_ids["id"].nunique())

duplicate_ids[
    ["id", "version_id", "title", "employer", "published"]
].head(10)


Antal rader som tillhör ett duplicerat ID: 8675
Antal duplicerade ID: 4120


,id,version_id,title,employer,published
478,29354585,f0883b87209ad585259381a396b05e5618ea3d93,Projektledare,Randstad AB,2025-01-02T10:15:14
477,29354585,26008c8a15f486a5b9163029fddd5d7fc3c03cb4,Projektledare,Randstad AB,2025-01-02T10:15:14
452,29361752,27b784e84eb1a5703befb304f9f76c3a3c8797db,Teknikansvarig VA till SYSTRA AB i Jönköping,SYSTRA AB,2025-01-07T13:27:42
451,29361752,5fddf1db951f3374f7792910c4f425650ce236e6,Teknikansvarig VA till SYSTRA i Jönköping,SYSTRA AB,2025-01-07T13:27:42
445,29362368,aa58e8f5ebbe56dd9dbc558eea70eb14085eb6de,Lösningsarkitekt Nätverksinfrastruktur,Försvarsmakten,2025-01-07T15:11:03
444,29362368,98e4dc9c1ec0419c0fcef0319b64909342df680d,Lösningsarkitekt Nätverksinfrastruktur,Försvarsmakten,2025-01-07T15:11:03
443,29362443,a335ce8a599380486cb101644acedcc27bef5d5d,Lösningsarkitekt IT-säkerhet,Försvarsmakten,2025-01-07T15:28:34
442,29362443,4acb1a3da382edaf553e92b57524ad381d704d85,Lösningsarkitekt IT-säkerhet,Försvarsmakten,2025-01-07T15:28:34
431,29363892,fdde3617b56e6d11100f6ad997d8606bd3dfbbc5,Senior Frontend Engineer,Schibsted Sverige AB,2025-01-08T09:56:04
430,29363892,de9e985fb72fb75014b4c4f676ce7c8bee93db9e,Senior Frontend Engineer,Schibsted Sverige AB,2025-01-08T09:56:04


In [28]:
html_pattern = r"<[^>]+>"

html_ads = df[
    df["description"]
    .fillna("")
    .str.contains(html_pattern, regex=True)
]

print("Antal annonser med möjlig HTML:", len(html_ads))

html_ads[
    ["id", "title", "description"]
].head(10)

Antal annonser med möjlig HTML: 6


,id,title,description
6986,30542907,Infrastrukturarkitekt till Växjö kommun,Vill du arbeta med teknik som gör verklig skil...
7143,30542907,Infrastrukturarkitekt till Växjö kommun,Vill du arbeta med teknik som gör verklig skil...
7607,30622455,Chefsarkitekt | Jula | Skara,Är du redo att forma Julas digitala framtid? D...
8086,30622455,Chefsarkitekt | Jula | Skara,Är du redo att forma Julas digitala framtid? D...
9293,30911356,Lösningsarkitekt,Vill du vara med och forma framtidens digitala...
9511,30911356,Lösningsarkitekt,Vill du vara med och forma framtidens digitala...


In [29]:
personnummer_pattern = r"\b(?:19|20)?\d{6}[-+ ]?\d{4}\b"

personnummer_count = df["description"].fillna("").str.contains(
    personnummer_pattern, regex=True
).sum()

print("Annonser med möjliga personnummer:", personnummer_count)

Annonser med möjliga personnummer: 0


In [30]:
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

email_ads = df[
    df["description"]
    .fillna("")
    .str.contains(email_pattern, regex=True)
]

print("Antal annonser med e-post:", len(email_ads))


Antal annonser med e-post: 0


In [31]:
phone_pattern = r"(?<!\d)(?:\+46|0046|0)[\s-]?(?:\d[\s-]?){7,11}(?!\d)"

phone_ads = df[
    df["description"]
    .fillna("")
    .str.contains(phone_pattern, regex=True)
]

print("Antal annonser med möjliga telefonnummer:", len(phone_ads))


Antal annonser med möjliga telefonnummer: 35


In [32]:
phone_matches = df["description"].fillna("").str.findall(phone_pattern)

phone_results = df.loc[
    phone_matches.str.len() > 0,
    ["id", "title", "employer"]
].copy()

phone_results["phone_match"] = phone_matches[
    phone_matches.str.len() > 0
].apply(lambda x: ", ".join(x))

phone_results

,id,title,employer,phone_match
79,29422415,Senior Kravanalytiker,Deploja AB,03-02-2025\n
295,29386905,IT-arkitekt till domänen Business Common,Vattenfall AB,02-02-2025
296,29386904,IT-arkitekt till domänen Business Common,Vattenfall AB,02-02-2025
297,29386902,IT-arkitekt till domänen Business Common,Vattenfall AB,02-02-2025
882,29439711,"IT-arkitekt, inriktning Digital Arbetsplats",Länsstyrelsen i Västra Götalands län,000-59 000
1232,29533240,Dataanalytiker sökes av Göteborgs Spårvägar,Göteborgs Stads Kollektivtrafik AB,000-52 000
1605,29663343,IT Specialist till VB Energi,AFRY AB,0240-876 00
2100,29663343,IT Specialist till VB Energi,AFRY AB,0240-876 00
3021,29791470,Senior IT-arkitekt,Deploja AB,01-07-2025
3313,29791470,Senior IT-arkitekt,Deploja AB,01-07-2025


In [33]:
phone_matches = df["description"].fillna("").str.findall(phone_pattern)

phone_ads = df.loc[
    phone_matches.str.len() > 0
].copy()

for _, row in phone_ads.iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])
    print("Träff:", phone_matches.loc[row.name])
    
    description = row["description"]
    
    for match in phone_matches.loc[row.name]:
        start = description.find(match)
        context = description[max(0, start - 100): start + len(match) + 100]
        print("Sammanhang:")
        print(context)

ID: 29422415
Titel: Senior Kravanalytiker
Träff: ['03-02-2025\n']
Sammanhang:
n av in- och utbetalningar• Arbetslivserfarenhet av arbete på statlig myndighetSista ansökningdatum:03-02-2025
Om DeplojaVi sysselsätter idag erfarna konsulter, inom bl.a. Systemutveckling, Test, Arkitektur, IT-
ID: 29386905
Titel: IT-arkitekt till domänen Business Common
Träff: ['02-02-2025']
Sammanhang:
 föräldraledighetstillägg med mera. Läs mer om våra förmåner här
. Välkommen med din ansökan senast 02-02-2025. Urval och intervjuer sker löpande. Vi tar endast emot ansökningar via vår webbplats. Vi är övertyg
ID: 29386904
Titel: IT-arkitekt till domänen Business Common
Träff: ['02-02-2025']
Sammanhang:
 föräldraledighetstillägg med mera. Läs mer om våra förmåner här
. Välkommen med din ansökan senast 02-02-2025. Urval och intervjuer sker löpande. Vi tar endast emot ansökningar via vår webbplats. Vi är övertyg
ID: 29386902
Titel: IT-arkitekt till domänen Business Common
Träff: ['02-02-2025']
Sammanhang:
 fö

## Cleaning

In [34]:
print("Antal annonser före rensning:", len(df))
print("Antal unika ID före rensning:", df["id"].nunique())

df_clean = df.drop_duplicates(subset="id", keep="first").copy()


print("-------------------------------------------")
print("Antal annonser efter rensning:", len(df_clean))
print("Antal unika ID efter rensning:", df_clean["id"].nunique())

Antal annonser före rensning: 11204
Antal unika ID före rensning: 6649
-------------------------------------------
Antal annonser efter rensning: 6649
Antal unika ID efter rensning: 6649


In [35]:
missing_description = df_clean[
    df_clean["description"].isna() |
    df_clean["description"].fillna("").str.strip().eq("")
]

print("Annonser utan description efter rensning:", len(missing_description))

Annonser utan description efter rensning: 8


In [36]:
df_clean = df_clean[
    df_clean["description"].fillna("").str.strip().ne("")
].copy()

print("Antal annonser efter borttagning:", len(df_clean))

Antal annonser efter borttagning: 6641


In [37]:
phone_ads_after_duplicates = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(phone_pattern, regex=True)
]

print(
    "Annonser med möjlig telefonträff efter dublettrensning:",
    len(phone_ads_after_duplicates)
)

Annonser med möjlig telefonträff efter dublettrensning: 23


In [38]:
def clean_phone_numbers(text):
    if pd.isna(text):
        return text
    
    return re.sub(phone_pattern, "[PHONE]", text)


df_clean["description"] = df_clean["description"].apply(clean_phone_numbers)


phone_matches_clean = df_clean["description"].fillna("").str.findall(phone_pattern)

print(
    "Antal möjliga telefonträffar efter cleaning:",
    phone_matches_clean.str.len().sum()
)

Antal möjliga telefonträffar efter cleaning: 0


In [39]:
phone_replacements = df_clean["description"].fillna("").str.count(r"\[PHONE\]").sum()

print("Antal ersatta telefonnummer:", phone_replacements)

Antal ersatta telefonnummer: 23


In [40]:
html_ads = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(html_pattern, regex=True)
]

print("Antal HTML-annonser:", len(html_ads))

Antal HTML-annonser: 3


In [43]:
for _, row in html_ads.iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])
    print(row["description"])

ID: 30542907
Titel: Infrastrukturarkitekt till Växjö kommun
Vill du arbeta med teknik som gör verklig skillnad? Hos oss får du chansen att påverka utvecklingen av en hel kommun, där smarta digitala lösningar bidrar till en enklare, tryggare och mer hållbar vardag för invånare, företag och besökare. Nu söker vi en infrastrukturarkitekt som vill vara en nyckelspelare i vår tekniska utvecklingsresa och som motiveras av att arbeta i ett komplext och samhällsviktigt IT‑landskap. Växjö kommun är Kronobergs största arbetsgivare och varje dag arbetar vi tillsammans för framtidens samhälle. <img src="https://sae3ndp007.blob.core.windows.net/swe/71d22a21-1a8c-479b-8fac-9af5f5be811b.jpg">
 
 
Om rollen 
Som infrastrukturarkitekt blir du en del av vårt team Arkitektur & Utveckling, tillsammans med 12 kollegor inom infrastruktur-, nät- och lösningsarkitektur samt systemutveckling. Här får du en central roll i att forma vår tekniska riktning och skapa robusta och säkra lösningar. Du får arbeta i en 

In [44]:
def clean_html(text):
    if pd.isna(text):
        return text
    
    return BeautifulSoup(text, "html.parser").get_text(" ", strip=True)


df_clean["description"] = df_clean["description"].apply(clean_html)


html_after = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(r"<[^>]+>", regex=True)
]

print("Annonser med HTML efter cleaning:", len(html_after))

Annonser med HTML efter cleaning: 0


- Vid kontroll av potentiella personuppgifter identifierades inga träffar för e-postadresser eller personnummer.  
 
- Regexbaserad identifiering av telefonnummer gav 35 möjliga träffar. Dessa granskades manuellt för att bedöma om de utgjorde faktiska telefonnummer.   
Några av träffarna visade sig vara telefonnummer till företagsväxlar. Verifierade telefonnummer, inklusive företagsväxlar, ersätts med `[PHONE]` eftersom de inte är relevanta för kompetensanalysen.

- Efter rensning av dubletter identifierades 8 annonser som saknade användbar `description`. Dessa annonser tas bort eftersom de inte innehåller någon text som kan användas i kompetensanalysen. 

- Förekomst av HTML-taggar kontrollerades med regex. Ett fåtal annonser innehöll HTML-formatering, men mängden var begränsad. HTML-taggarna tas bort eftersom de inte innehåller relevant information för kompetensanalysen och annars kan utgöra brus vid den fortsatta NLP-bearbetningen.

- Regexbaserad identifiering kan användas för strukturerade uppgifter som e-postadresser, telefonnummer och personnummer men är mindre lämplig för att identifiera personnamn. Förekomst av personnamn kommer därför undersökas separat med spaCy NER. 



In [47]:
contact_pattern = (
    r"(?i)(kontaktperson|kontakt|kontakta|ansvarig|rekryterare|"
    r"rekryteringsansvarig|för frågor|vid frågor|har du frågor)"
)

contact_ads = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(contact_pattern, regex=True)
]

print("Annonser med kontaktfraser:", len(contact_ads))

for _, row in contact_ads.head(20).iterrows():
    description = row["description"]
    match = re.search(contact_pattern, description)

    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])

    if match:
        start = match.start()
        end = min(len(description), match.end() + 200)

        print(description[start:end].replace("\n", " "))

C:\Users\magda\AppData\Local\Temp\ipykernel_18936\510374362.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(contact_pattern, regex=True)


Annonser med kontaktfraser: 1749
ID: 29427257
Titel: IT-konsult
kontakta vem som helst av dina kollegor i världen med vilken fråga du än har låter som en klyscha, men så ser vardagen ut på Capgemini. Vi är varandras bästa coacher. Arbetsuppgifter: Som IT-konsult arbetar 
ID: 29430361
Titel: Förvaltningsledare IT till Nexer
kontakter och avtal   Personliga egenskaper  Att du är lyhörd och har ett starkt driv med en förmåga att skapa struktur är viktigt för oss. Samtidigt ser vi att du är lösningsorienterad och flexibel i ditt s
ID: 29429018
Titel: Junior Data and Financial Analyst till Modity
kontaktytor med andra avdelningar som handel, IT, portföljförvaltning och sälj, vilket kräver att du är kommunikativ och har lätt för att samarbeta. Kombinationen av IT, datalager och ekonomi gör rollen unik
ID: 29428116
Titel: Affärsutvecklare Tech till Kalmar Science Park 
kontakt med olika företagsledare, eller att hosta ett event i något av våra nätverk. Samtidigt är du en doer som är van vid a